In [ ]:
import sys
import os
import itertools

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
import torch 

from auto_circuit.data import load_datasets_from_json
from auto_circuit.experiment_utils import load_tl_model
from auto_circuit.prune_algos.mask_gradient import mask_gradient_prune_scores
from auto_circuit.types import PruneScores
from auto_circuit.utils.graph_utils import patchable_model, patch_mode
from auto_circuit.utils.misc import repo_path_to_abs_path
from auto_circuit.visualize import draw_seq_graph
from auto_circuit.utils.ablation_activations import src_ablations
from auto_circuit.types import (
    AblationType,
    CircuitOutputs,
    PatchType,
)
from auto_circuit.utils.tensor_ops import (
    correct_answer_proportion, 
    correct_answer_greater_than_incorrect_proportion, 
    batch_avg_answer_diff, 
    batch_answer_diff_percents, 
    correct_answer_greater_than_incorrect_proportion
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import PreTrainedTokenizerFast, AutoTokenizer
import transformer_lens as tl
from transformer_lens import HookedTransformer, HookedTransformerConfig
import json

%load_ext autoreload
%autoreload 2

In [ ]:
TOKENIZER_DIR  = "../model/wordlevel_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, add_bos_token=True)

MODEL = 'full_model_rope'
# --- Load config ---
with open(f"../model/{MODEL}/config.json", "r") as f:
    cfg_dict = json.load(f)

# --- Fix dtype string back to actual torch dtype ---
if isinstance(cfg_dict.get("dtype"), str):
    cfg_dict["dtype"] = getattr(torch, cfg_dict["dtype"].replace("torch.", ""))

# --- Rebuild config and model ---
config = HookedTransformerConfig.from_dict(cfg_dict)
model = HookedTransformer(config)
model.load_state_dict(torch.load(f"../model/{MODEL}/model_weights.pth"))
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

model.set_tokenizer(tokenizer)

model.cfg.default_prepend_bos, model.cfg.tokenizer_prepends_bos

model.set_use_attn_result(True)
model.set_use_attn_in(True)
model.set_use_split_qkv_input(True)
if hasattr(model.cfg, "use_hook_mlp_in"):
    model.set_use_hook_mlp_in(True) 

model.eval()

for param in model.parameters():
    param.requires_grad = False

# make save path
if not os.path.exists(repo_path_to_abs_path(f"/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/{MODEL}")):
    os.makedirs(repo_path_to_abs_path(f"/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/{MODEL}"))
    
SAVE = False

In [ ]:
path = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_last_big.json")

train_loader_last, test_loader_last = load_datasets_from_json(
    model=model,
    path=path,
    device=device,
    prepend_bos=False,
    tail_divergence=False,
    batch_size=32,
    train_test_size=(256, 256),
)

auto_model_last = patchable_model(
    model,
    factorized=True,
    slice_output="last_seq",
    separate_qkv=True,
    device=device,
)

attribution_scores_last: PruneScores = mask_gradient_prune_scores(
    model=auto_model_last,
    dataloader=train_loader_last,
    official_edges=None,
    grad_function="logit",
    answer_function="avg_diff",
    mask_val=0.0,
)

fig = draw_seq_graph(
    auto_model_last, attribution_scores_last, score_threshold=3.5, layer_spacing=True, orientation="v"
)
if SAVE:
    fig.write_image(repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/test_last_2l.png"), scale=4)

In [ ]:
tl.utils.test_prompt(prompt="the last term in the sequence 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 is", answer= '20', model=model, print_details=True)

In [ ]:
from utils.circuit_discovery import get_real_edges

edges_A, remaining_edges_A = get_real_edges(
    model,
    attribution_scores_last,
    score_threshold=4.5,
    print_egdes=True,
    return_edges=True,
)

In [ ]:
path = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_next_big.json")

train_loader_next, test_loader_next = load_datasets_from_json(
    model=model,
    path=path,
    device=device,
    prepend_bos=False,
    tail_divergence=False,
    batch_size=32,
    train_test_size=(256, 256),
)

auto_model_next = patchable_model(
    model,
    factorized=True,
    slice_output="last_seq",
    separate_qkv=True,
    device=device,
)

attribution_scores_next: PruneScores = mask_gradient_prune_scores(
    model=auto_model_next,
    dataloader=train_loader_next,
    official_edges=None,
    grad_function="logit",
    answer_function="avg_diff",
    mask_val=0.0,
)

fig = draw_seq_graph(
    auto_model_next, attribution_scores_next, score_threshold=4.5, layer_spacing=True, orientation="v"
)
if SAVE:
    fig.write_image(repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/test_next_2l.png"), scale=4)

In [ ]:
tl.utils.test_prompt(prompt="the next term in the sequence 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 is", answer= '20', model=model, print_details=True)

In [ ]:
from utils.circuit_discovery import compute_circuit_overlap

edges_B, remaining_edges_B = get_real_edges(
    model,
    attribution_scores_next,
    score_threshold=4.5,
    print_egdes=True,
    return_edges=True,
)

In [ ]:
metrics = compute_circuit_overlap(
    edges_A,
    edges_B,
    attribution_scores_A=attribution_scores_last,
    attribution_scores_B=attribution_scores_next,
    print_diffs=True
)

print("\n=== Summary Metrics ===")
print(f"Edge‐set Intersection: {metrics['edge_intersection']}")
print(f"Edge‐set Union:       {metrics['edge_union']}")
print(f"Edge Jaccard Index:   {metrics['edge_jaccard']:.3f}")
print(f"Node‐set Intersection: {metrics['node_intersection']}")
print(f"Node‐set Union:       {metrics['node_union']}")
print(f"Node Jaccard Index:   {metrics['node_jaccard']:.3f}")

In [ ]:
from utils.circuit_discovery import save_comparison_plot
path_A = f"../auto_circuit_exps/{MODEL}/last_circuit.png"
path_B = f"../auto_circuit_exps/{MODEL}/next_circuit.png"

if SAVE:
    save_comparison_plot(
        path_A,
        path_B,
        remaining_edges_A,
        remaining_edges_B,
        output_path=repo_path_to_abs_path(f"/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/{MODEL}/comparison.png"),
)

In [ ]:
auto_model = patchable_model(
    model,
    factorized=True,
    slice_output="last_seq",
    separate_qkv=True,
    kv_caches=(train_loader_last.kv_cache, test_loader_last.kv_cache), # this is needed else, patching will not work
    device=device,
)

for batch in test_loader_last:
    toks = batch.clean
    answers = batch.answers
    wrong_answers = batch.wrong_answers
    answers = [(answers[i], wrong_answers[i]) for i in range(len(answers))]
    answers = torch.tensor(answers, dtype=torch.long)
    answers = answers.to(device)

ablations = src_ablations(auto_model, toks, AblationType.ZERO)

# target_edge_str = "A1.0->Resid End" # test if an edge is present
# all_edges = list(itertools.chain.from_iterable(auto_model.edge_dict.values()))
all_edge_strs = [str(edge) for edge in edges_A] # these are desceonding order by attribution score
edge_str = all_edge_strs[0]  # e.g., "MLP 1->Resid End"
found = any(
    edge_str in subdict
    for subdict in auto_model.edge_name_dict.values()
)
assert found, f"Edge '{edge_str}' not found in edge_name_dict"

# patch_edges = [edge for edge in all_edge_strs if edge != target_edge_str]  # patch all edges except the target edge
patch_edges = [edge for edge in all_edge_strs if edge != all_edge_strs[0]]  # patch all edges except the edge with the highest attribution score
print(f"Edges to patch: {patch_edges}")
with patch_mode(auto_model, ablations, patch_edges):
    patched_out = auto_model(toks)

from utils.direct_logit_attribution import logits_to_ave_logit_diff
ave_logit_diff = logits_to_ave_logit_diff(patched_out, answer_tokens=answers, per_prompt=False)
print("Average Logit Difference:", ave_logit_diff)

In [ ]:
with patch_mode(auto_model, ablations, patch_edges):
    for batch in test_loader_last:
        patched_out = auto_model(batch.clean)

for batch in test_loader_last:
    clean_out = model(batch.clean)

proportion_correct = correct_answer_proportion(patched_out[:, -1, :], batch)
print("Proportion of correct answers after patching:", proportion_correct)

correct_answer_greater_than_incorrect_proportion_value = correct_answer_greater_than_incorrect_proportion(patched_out[:, -1, :], batch)
print("Proportion of correct answers greater than incorrect after patching:", correct_answer_greater_than_incorrect_proportion_value)

batch_avg_answer_diff_value = batch_avg_answer_diff(patched_out[:, -1, :], batch)
print("Average answer difference after patching:", batch_avg_answer_diff_value)

batch_avg_answer_diff_value_clean = batch_avg_answer_diff(clean_out[:, -1, :], batch)
print("Average answer difference full model:", batch_avg_answer_diff_value_clean)

batch_answer_diff_percents_value = batch_answer_diff_percents(patched_out[:, -1, :], clean_out[:, -1, :], batch)
print("Batch answer difference percents after patching:", batch_answer_diff_percents_value)

In [ ]:
cands = [(layer, head) for layer in range(model.cfg.n_layers) for head in range(model.cfg.n_heads)]
# Compute the full Composition-Score tensor for OV→Q 
#    mode="Q" means OV ∘ Q, i.e. OV writes → Q reads
CS_Q = model.all_composition_scores("Q")  
CS_K = model.all_composition_scores("K")  
CS_V = model.all_composition_scores("V")  

CS = torch.stack([CS_Q, CS_K, CS_V], dim=0)  # shape [3, L, H, L, H]

# For each candidate Q/K/V-head (l2,h2), find its best OV partner
modes = ['Q', 'K', 'V'] 
best_pairs = []
for i in modes:
    print(f"Composition scores for mode {i}:")
    id = modes.index(i)  
    for (l2, h2) in cands:
        scores = CS[id, :, :, l2, h2]  # shape [L, H]
        flat = scores.flatten()
        max_idx = flat.argmax().item()
        l1 = max_idx // model.cfg.n_heads
        h1 = max_idx % model.cfg.n_heads
        best_pairs.append(((l1, h1), (l2, h2), flat[max_idx].item()))

    # Sort by descending CS and inspect
    best_pairs.sort(key=lambda x: x[2], reverse=True)
    for (l1,h1), (l2,h2), score in best_pairs:
        print(f"OV head {(l1,h1)} → {i} head {(l2,h2)} has CS = {score:.3f}")
